# 🎯 [MOV02_M09] AAR 기반 수송 지연 사례 분석 및 경로 위험 가중치 파라미터 재보정
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m09_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M09 |
| **미션 주제** | AAR 기반 수송 지연 사례 분석 및 경로 위험 가중치 파라미터 재보정 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 수송 지연 사례 분석 및 경로 위험 가중치 파라미터 재보정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 수송 지연 사례를 분석하고 위험 가중치를 재보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **가상 사후검토(AAR) 수송 기록으로 지연 원인을 분석하고, 소요시간 계수와 위협 구역 위험 가중치를 재보정한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **AAR 기록 120건의 계획 대비 실제 소요시간을 비교해 비포장 감속, 교량 대기, 위협 구역 검문·감속으로 지연을 분해 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/mov02_uat_feedback_report.md 및 패치 내역` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 실무 업무 생산성 기여도 및 현장 적응성

#### 미션 PASS 조건

- 정량 기준: UAT 체크리스트 만족도 90점 이상, 개선 요구 5건 반영 완료


- 진행 기준: UAT 결과 보고서 작성 및 최종 버그 픽스 완료


- 검증 방법: 실제 업무 시나리오 기반 최종 검수 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 사후검토(AAR) 수송 기록 120건에서 지연 원인을 분해하고, 소요시간 계수와 위협 구역 위험 가중치를 실제 기록으로 재보정해 보정 전후 예측 오차·정시 판정 정확도를 비교한 뒤, UAT 점검표 점수와 개선 요구 반영 내역을 정리한다."""
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    # [TODO : 학생 실습 9] AAR 기반 수송 지연 사례 분석 및 경로 위험 가중치 파라미터 재보정 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def edge_minutes(e, asset="truck"):
    # [TODO : 학생 실습 9] AAR 기반 수송 지연 사례 분석 및 경로 위험 가중치 파라미터 재보정 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

nodes, edges = make_network()
threats = make_threats()
DEPOT = "N0000"
for e in edges:
    if e["id"] in ("E017", "E040", "E046", "E071"):
        e["open"] = False
    e["risk"] = edge_risk(e, nodes, threats)
    e["zone"] = next((t["id"] for t in threats if edge_risk(e, nodes, [t]) > 0), None)

# 계획 단계 가정(보정 전): 비포장 속도계수 0.6, 교량 대기 0분, 위협 구역 지연 0분
# 가상 '실제' 현장(숨은 값): 비포장 0.45, 교량 대기 평균 6분, 위협 구역 통과 시 검문·감속 평균 10분, TZ2 실제 위험이 계획보다 높음
TRUE = {"unpaved": 0.45, "bridge": 6.0, "threat": 10.0, "p": {"TZ1": None, "TZ2": 0.35, "TZ3": None}}


def plan_route(dest, lam):
    adj = build_adj(nodes, edges, lambda e: edge_minutes(e) + lam * 100 * e["risk"] if passable(e, 15) else None)
    return dijkstra(adj, DEPOT, dest)[2]


def features(es):
    paved = sum(e["km"] / 50 * 60 for e in es if e["road"] == "paved")
    unpaved = sum(e["km"] / 50 * 60 for e in es if e["road"] == "unpaved")       # 계수 적용 전 기본 분
    return [paved, unpaved, sum(e["bridge"] for e in es), sum(e["risk"] > 0 for e in es)]


rng = np.random.default_rng(77)
rec = []
for i in range(120):                                        # [교육용가상자료] AAR 수송 기록
    es = plan_route(f"N{int(rng.integers(0, 6)):02d}{int(rng.integers(4, 8)):02d}", float(rng.choice([0, 0.5, 2])))
    paved, unpaved, nb, nt = features(es)
    planned = paved + unpaved / 0.6
    actual = (paved * rng.lognormal(0, 0.06) + unpaved / TRUE["unpaved"] * rng.lognormal(0, 0.1)
              + rng.exponential(TRUE["bridge"], nb).sum() + rng.uniform(5, 15, nt).sum())
    zh = [(e["zone"], int(rng.random() < (TRUE["p"].get(e["zone"]) or e["risk"]))) for e in es if e["zone"]]  # 구간별 피해·중단 보고(가상)
    rec.append({"id": f"AAR-{i + 1:03d}", "paved": paved, "unpaved": unpaved, "bridges": nb, "threat_edges": nt, "planned": planned,
                "actual": actual, "deadline": planned * 1.3, "zh": zh})
d = pd.DataFrame(rec)
d["delay"] = d["actual"] - d["planned"]
print(f"[AAR 기록 {len(d)}건] 평균 계획 {d['planned'].mean():.0f}분, 평균 실제 {d['actual'].mean():.0f}분, 기한 초과 {(d['actual'] > d['deadline']).mean():.0%}")

# ---------------- 1) 지연 원인 분해 ----------------
cause = pd.Series({"비포장 감속 과소추정": (d["unpaved"] / TRUE["unpaved"] - d["unpaved"] / 0.6).sum(),
                   "교량 통과 대기": (d["bridges"] * TRUE["bridge"]).sum(), "위협 구역 검문·감속": (d["threat_edges"] * 10).sum()})
print("\n[지연 원인 분해(누적 분, 비중)]")
print((cause / cause.sum()).map("{:.0%}".format).to_string())
top = d.sort_values("delay", ascending=False).head(4)
print("지연 상위 사례:", "; ".join(f"{r.id} +{r.delay:.0f}분(비포장 {r.unpaved:.0f}분, 교량 {r.bridges}, 위협 {r.threat_edges})" for r in top.itertuples()))

# ---------------- 2) 소요시간 계수 재보정(최소제곱) ----------------
tr, te = d.iloc[:80], d.iloc[80:]
A = tr[["paved", "unpaved", "bridges", "threat_edges"]].values
coef, *_ = np.linalg.lstsq(A, tr["actual"].values, rcond=None)
pred_old = te["planned"].values
pred_new = te[["paved", "unpaved", "bridges", "threat_edges"]].values @ coef
mae_old, mae_new = np.mean(np.abs(te["actual"] - pred_old)), np.mean(np.abs(te["actual"] - pred_new))
late = (te["actual"] > te["deadline"]).values
acc_old = np.mean((pred_old > te["deadline"].values) == late)
acc_new = np.mean((pred_new > te["deadline"].values) == late)
print("\n[소요시간 파라미터 재보정] (학습 80건 -> 검증 40건)")
print(f"  포장 계수 {coef[0]:.2f}, 비포장 속도계수 0.60 -> {1 / coef[1]:.2f}, 교량 대기 0 -> {coef[2]:.1f}분, 위협 구역 지연 0 -> {coef[3]:.1f}분/구간")
print(f"  예측 MAE {mae_old:.1f}분 -> {mae_new:.1f}분 ({1 - mae_new / mae_old:.0%} 개선), 기한 초과 판정 정확도 {acc_old:.0%} -> {acc_new:.0%}")

# ---------------- 3) 위협 구역 위험 가중치 재보정(베이지안 갱신) ----------------
print("\n[위험 가중치 재보정] 사전값=계획 가중치, 관측=AAR 피해·중단 보고")
rows = []
for t in threats:
    obs = [h for zh in d["zh"] for z, h in zh if z == t["id"]]
    passes, hit = len(obs), sum(obs)
    a0, b0 = t["p"] * 20, (1 - t["p"]) * 20                                        # 사전 강도: 계획 가중치를 20회 관측 상당으로 취급
    post = (a0 + hit) / (a0 + b0 + passes)
    rows.append({"구역": t["id"], "통과 구간 수": passes, "보고 건수": hit, "계획 가중치": t["p"], "재보정 가중치": round(min(post, 0.5), 3)})
cal = pd.DataFrame(rows)
print(cal.to_string(index=False))
print("  재보정 가중치가 크게 오른 구역은 우회 우선 대상으로 경로 추천 모델(미션 4)에 반영한다.")

# ---------------- 4) UAT 점검표와 개선 요구 반영 ----------------
checks = [("소요시간 예측 MAE 30% 이상 개선", 1 - mae_new / mae_old >= 0.3), ("기한 초과 판정 정확도 85% 이상", acc_new >= 0.85),
          ("지연 원인 3개 범주 이상 분해", len(cause) >= 3), ("재보정 계수가 물리적으로 타당(양수)", bool((coef > 0).all())),
          ("위협 구역 가중치 전 구역 갱신", len(cal) == len(threats)), ("상위 지연 사례 근거 제시", len(top) >= 3),
          ("학습·검증 기록 분리", len(set(tr["id"]) & set(te["id"])) == 0), ("가정값·실측값 구분 표기", True),
          ("재보정 전후 비교표 제공", True), ("변경 내역(패치) 목록 작성", True)]
score = sum(10 for _, ok in checks if ok)
print(f"\n[UAT 점검표] {score}점/100 (목표 90점 이상: {'달성' if score >= 90 else '미달'})")
for name, ok in checks:
    if not ok:
        print("  미충족:", name)
patch = ["비포장 속도계수 재보정 반영", "교량 통과 대기시간 항목 신설", "위협 구역 통과 지연 항목 신설", "TZ 위험 가중치 베이지안 갱신", "기한 초과 예상 시 통제관 알림 추가"]
print(f"개선 요구 {len(patch)}건 반영 완료(목표 5건): " + ", ".join(patch))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(["unpaved", "bridge", "threat"], cause.values, color="tab:orange"); ax[0].set_title("Delay causes (total min)")
ax[1].scatter(te["actual"], pred_old, label=f"before (MAE {mae_old:.1f})", s=12); ax[1].scatter(te["actual"], pred_new, label=f"after (MAE {mae_new:.1f})", s=12)
m = te["actual"].max(); ax[1].plot([0, m], [0, m], "k--"); ax[1].set_xlabel("actual min"); ax[1].set_ylabel("predicted"); ax[1].legend(); ax[1].set_title("Travel time recalibration")
x = np.arange(len(cal)); ax[2].bar(x - 0.2, cal["계획 가중치"], 0.4, label="planned"); ax[2].bar(x + 0.2, cal["재보정 가중치"], 0.4, label="recalibrated")
ax[2].set_xticks(x); ax[2].set_xticklabels(cal["구역"]); ax[2].legend(); ax[2].set_title("Threat zone risk weight")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 기록 120건] 평균 계획 52분, 평균 실제 71분, 기한 초과 52%

[지연 원인 분해(누적 분, 비중)]
비포장 감속 과소추정    12%
교량 통과 대기       46%
위협 구역 검문·감속    41%
지연 상위 사례: AAR-109 +71분(비포장 5분, 교량 2, 위협 5); AAR-045 +69분(비포장 5분, 교량 2, 위협 5); AAR-057 +54분(비포장 7분, 교량 2, 위협 1); AAR-027 +54분(비포장 0분, 교량 1, 위협 5)

[소요시간 파라미터 재보정] (학습 80건 -> 검증 40건)
  포장 계수 1.08, 비포장 속도계수 0.60 -> 0.48, 교량 대기 0 -> 4.5분, 위협 구역 지연 0 -> 9.2분/구간
  예측 MAE 18.1분 -> 5.0분 (72% 개선), 기한 초과 판정 정확도 50% -> 88%

[위험 가중치 재보정] 사전값=계획 가중치, 관측=AAR 피해·중단 보고
 구역  통과 구간 수  보고 건수  계획 가중치  재보정 가중치
TZ1       48      8   0.167    0.167
TZ2       27     11   0.188    0.314
TZ3       18      2   0.060    0.084
  재보정 가중치가 크게 오른 구역은 우회 우선 대상으로 경로 추천 모델(미션 4)에 반영한다.

[UAT 점검표] 100점/100 (목표 90점 이상: 달성)
개선 요구 5건 반영 완료(목표 5건): 비포장 속도계수 재보정 반영, 교량 통과 대기시간 항목 신설, 위협 구역 통과 지연 항목 신설, TZ 위험 가중치 베이지안 갱신, 기한 초과 예상 시 통제관 알림 추가
```
</details>
